# Notebook 07 — Người 3 (Tuần 11): DeBERTa Prompt-Aware + chấm Official Test một lần cho L0 / L1 / Prompt-Aware

**Nhóm 06 — CLC02 — PERSUADE 2.0 Discourse Effectiveness**

Notebook này thay cho `script_improved_deberta_colab.py` (script cũ chỉ in một dòng chữ, không train, không ghi kết quả) và cho các số DeBERTa đang được gõ cứng trong `script_official_test_evaluation.py`.

| Bước | Làm gì | Thời gian ước tính (T4 / A100) |
|---|---|---|
| 1 | Cài thư viện, mount Drive, kiểm tra GPU | 2 phút |
| 2 | Tải / tìm dữ liệu (train + test) | 2–5 phút |
| 3 | Dựng 3 kiểu đầu vào L0, L1, Prompt-Aware (PA) — PA = L1 + tên đề, **chỉ đổi đầu vào** | 1 phút |
| 4 | Train PA trên đúng split 80/20 cũ, cùng siêu tham số với L0/L1 (resume được nếu Colab ngắt) | ~4–6 giờ / ~1–1.5 giờ |
| 5 | Tìm checkpoint L0 & L1 cũ; nếu không có thì train lại bằng đúng công thức | 0 hoặc thêm ~4–6 giờ mỗi model |
| 6 | Validation: kiểm tra checkpoint L0/L1 tái lập đúng số cũ, so PA vs L1 (bootstrap theo bài) | ~15–30 phút |
| 7 | **OFFICIAL TEST — chạy 1 lần duy nhất** cho L0, L1, PA cùng lượt, có khóa chống chạy lại | ~30–60 phút |
| 8 | Xuất CSV cho bảng Master + bảng tổng hợp 5 hệ thống | 1 phút |

**Kỷ luật test (theo nhận xét Tuần 10 của Thầy Khánh, mục 2):** mọi quyết định (chọn model, siêu tham số) phải xong ở Bước 6 trên validation. Bước 7 ghi file khóa `OFFICIAL_TEST_LOCK.json`; chạy lại notebook sẽ chỉ đọc lại dự đoán đã lưu, không suy luận lại.

**Giữ lại kết quả chạy:** mọi CSV/JSON/log được ghi vào Drive (`MyDrive/MLA_Dataset/outputs/week11_person3/`). Sau khi chạy xong: *File → Save a copy in Drive* (hoặc *Download .ipynb*) để giữ lại output từng cell.

## Bước 0 — Chuẩn bị trên Google Drive (làm 1 lần)

Tạo thư mục `MyDrive/MLA_Dataset/` gồm:

```
MLA_Dataset/
├── persuade_corpus_2.0_train.csv        # BẢN CỦA NHÓM (sha256 f61319ed...) — nên upload từ máy, để khớp tuần 9–10
├── persuade_corpus_2.0_test.csv         # nếu chưa có, notebook tự tải từ link trong README GitHub scrosseye/persuade_corpus_2.0
└── outputs/
    ├── split_assignments.csv            # BẮT BUỘC — split 80/20 cũ (12.474 / 3.119 bài)
    ├── deberta_best_model/              # (nếu còn) checkpoint L0 từ script 02
    ├── deberta_context_best_model/      # (nếu còn) checkpoint L1 từ script 05
    ├── official_test_rf_actual_result.csv   # (tùy chọn) của Người 1 — để in bảng 5 hệ thống ngay trong notebook
    └── official_test_svm_actual_result.csv  # (tùy chọn) của Người 2
```

**Lưu ý thư viện:** Colab hiện cài sẵn transformers 5.x, bản này train DeBERTa-v3 bị hỏng (loss NaN / model đoán toàn một lớp — cũng là lý do checkpoint L1 Colab tuần 10 toàn NaN). Ô cài đặt đầu tiên tự hạ xuống 4.x rồi dừng; khi đó chọn *Runtime → Restart session* và chạy lại từ đầu.

Runtime: **Runtime → Change runtime type → GPU** (A100/L4 nếu có Colab Pro; T4 vẫn chạy được nhưng nên chia làm nhiều phiên — notebook resume được).

In [ ]:
# ===== BƯỚC 1: CÀI ĐẶT, IMPORT, KIỂM TRA GPU =====
import importlib.util, importlib.metadata, subprocess, sys
# transformers 5.x (Colab mặc định từ 10/2026) train DeBERTa-v3 hỏng: loss NaN / không học (đã kiểm chứng
# 28/10/2026: cùng cấu hình, 300 bước -> Macro-F1 0.29 ở 5.20.2 so với 0.51 ở 4.57.6). Bắt buộc dùng 4.x.
if importlib.util.find_spec("transformers") is None or int(importlib.metadata.version("transformers").split(".")[0]) >= 5:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.57,<5"])
    raise SystemExit("Đã cài transformers 4.x. Runtime -> Restart session, rồi chạy lại notebook từ đầu.")
for pkg, pip_name in [("datasets", "datasets"), ("accelerate", "accelerate"),
                      ("sentencepiece", "sentencepiece"), ("google.protobuf", "protobuf"), ("gdown", "gdown")]:
    if importlib.util.find_spec(pkg.split(".")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import os, gc, json, glob, time, random, shutil, hashlib, inspect, zipfile, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if USE_CUDA else "CPU"
# bf16 trên GPU hỗ trợ (A100/L4/H100): một lần chạy L1 fp16 trên Colab trước đây ra trọng số NaN
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()
USE_FP16 = USE_CUDA and not USE_BF16
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device: {GPU_NAME} | "
      f"precision: {'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'}")
if USE_CUDA:
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CẢNH BÁO: không có GPU — chỉ dùng được QUICK = True để chạy thử.")

torch 2.11.0+cu130 | transformers 4.57.6 | device: NVIDIA A100-SXM4-40GB | precision: bf16
VRAM: 39.5 GB


In [ ]:
# ===== CẤU HÌNH — chỉ sửa ô này =====
QUICK = False             # True = chạy thử vài chục bài, 1 epoch (KHÔNG dùng số này để báo cáo)
TRAIN_MISSING_BASELINES = True  # nếu không tìm thấy checkpoint L0/L1 cũ thì train lại bằng đúng công thức
RUN_OFFICIAL_TEST = False # đổi thành True SAU KHI đã xem kết quả validation ở Bước 6
MOUNT_DRIVE = False       # False khi chạy bằng colab CLI (drive.mount cần người bấm xác nhận)

# Siêu tham số: GIỮ NGUYÊN như script 02 (L0) và 05 (L1) để so sánh có kiểm soát
CFG = dict(
    model_name="microsoft/deberta-v3-base",
    max_length=512,
    batch_size=8,
    grad_accum=4,          # batch hiệu dụng 32
    lr=2e-5,
    epochs=3,
    warmup_ratio=0.1,
    seed=42,
    eval_batch_size=64,    # chỉ ảnh hưởng tốc độ suy luận
    n_bootstrap=2000,
)
if QUICK:
    CFG.update(epochs=1, max_length=256)

DRIVE_DIR = "/content/drive/MyDrive/MLA_Dataset"
# Link trong README của https://github.com/scrosseye/persuade_corpus_2.0 (test là zip có mật khẩu)
GDRIVE_TRAIN_ID = "13phHyDzIsb0MHyJr6q-B-qIa9P2tM135"
GDRIVE_TEST_ID = "1K1SIJiG-2zWgMlTzxQeYOcLwOsFaVel1"
TEST_ZIP_PASSWORD = "persuade_test"
EXPECTED_TRAIN_SHA256 = "f61319edd8bf16a982711ea0399fad59c05afaec05cdf0767f16a2c05c467e23"

In [ ]:
# ===== ĐƯỜNG DẪN, SEED, LOG =====
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

BASE_DIR = DRIVE_DIR if os.path.isdir(DRIVE_DIR) else os.getcwd()
OUT_DIR = os.path.join(BASE_DIR, "outputs")
P3_DIR = os.path.join(OUT_DIR, "week11_person3" + ("_QUICK" if QUICK else ""))
LOCAL_DATA_DIR = "/content/data" if IN_COLAB else os.path.join(BASE_DIR, "data_cache")
for d in (OUT_DIR, P3_DIR, LOCAL_DATA_DIR):
    os.makedirs(d, exist_ok=True)


def enforce_reproducibility(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if USE_CUDA:
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


enforce_reproducibility(CFG["seed"])

LOG_PATH = os.path.join(P3_DIR, "run_log.txt")


def log(msg):
    line = f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


log(f"BASE_DIR={BASE_DIR} | P3_DIR={P3_DIR} | QUICK={QUICK} | GPU={GPU_NAME}")

[2026-10-28 12:51:39 UTC] BASE_DIR=/content | P3_DIR=/content/outputs/week11_person3 | QUICK=False | GPU=NVIDIA A100-SXM4-40GB


## Bước 2 — Dữ liệu

- Train: ưu tiên bản của nhóm trên Drive (để khớp split và checkpoint tuần 9–10); nếu không có thì tải từ Google Drive gốc.
- Test: tìm `persuade_corpus_2.0_test.csv` hoặc file zip trên Drive; nếu không có thì tải zip gốc và giải nén với mật khẩu `persuade_test`.

**Lưu ý ID:** bản CSV của nhóm từng bị Excel lưu lại nên `essay_id` có dạng `5.40889E+12` và `discourse_id` gần như không còn duy nhất (728 giá trị). Vì vậy notebook:
- chuẩn hóa `essay_id` về dạng Excel (`{:.5E}`) để khớp `split_assignments.csv` dù dùng bản nào;
- dùng `essay_id_comp` (ID bài thật) để lấy câu trước/sau và để bootstrap theo bài;
- lưu dự đoán kèm `row_idx`, `essay_id_comp`, `discourse_start` để truy vết, không dựa vào `discourse_id`.

In [ ]:
# ===== BƯỚC 2: TÌM / TẢI DỮ LIỆU =====
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def gdown_file(file_id, dest):
    import gdown
    log(f"Tải từ Google Drive id={file_id} -> {dest}")
    gdown.download(id=file_id, output=dest, quiet=False)
    return dest


def find_train_csv():
    for p in [os.path.join(BASE_DIR, "persuade_corpus_2.0_train.csv"),
              os.path.join(BASE_DIR, "data", "persuade_raw", "persuade_corpus_2.0_train.csv"),
              os.path.join(LOCAL_DATA_DIR, "persuade_corpus_2.0_train.csv")]:
        if os.path.exists(p):
            return p
    return gdown_file(GDRIVE_TRAIN_ID, os.path.join(LOCAL_DATA_DIR, "persuade_corpus_2.0_train.csv"))


def extract_test_zip(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as zf:
            names = [n for n in zf.namelist() if n.endswith(".csv")]
            zf.extractall(LOCAL_DATA_DIR, pwd=TEST_ZIP_PASSWORD.encode())
    except (RuntimeError, NotImplementedError, zipfile.BadZipFile) as e:
        log(f"zipfile không giải nén được ({e}); thử 7z...")
        if shutil.which("7z") is None:
            subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)
        subprocess.run(["7z", "x", "-y", f"-p{TEST_ZIP_PASSWORD}", f"-o{LOCAL_DATA_DIR}", zip_path], check=True)
        names = []
    hits = glob.glob(os.path.join(LOCAL_DATA_DIR, "**", "*test*.csv"), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Giải nén {zip_path} xong nhưng không thấy CSV test. Trong zip: {names}")
    return hits[0]


def find_test_csv():
    for p in [os.path.join(BASE_DIR, "persuade_corpus_2.0_test.csv"),
              os.path.join(BASE_DIR, "data", "persuade_raw", "persuade_corpus_2.0_test.csv"),
              os.path.join(LOCAL_DATA_DIR, "persuade_corpus_2.0_test.csv")]:
        if os.path.exists(p):
            return p
    zips = glob.glob(os.path.join(BASE_DIR, "*test*.zip")) + glob.glob(os.path.join(LOCAL_DATA_DIR, "*test*.zip"))
    zip_path = zips[0] if zips else gdown_file(GDRIVE_TEST_ID, os.path.join(LOCAL_DATA_DIR, "persuade_corpus_2.0_test.zip"))
    return extract_test_zip(zip_path)


TRAIN_CSV = find_train_csv()
train_sha = sha256_of(TRAIN_CSV)
log(f"TRAIN_CSV={TRAIN_CSV} | sha256={train_sha}")
if train_sha != EXPECTED_TRAIN_SHA256:
    log("CHÚ Ý: train CSV khác bản nhóm dùng ở tuần 9–10 (sha256 khác). Split vẫn khớp nhờ chuẩn hóa essay_id, "
        "nhưng hãy ghi chú điều này trong báo cáo.")

SPLIT_CSV = os.path.join(OUT_DIR, "split_assignments.csv")
assert os.path.exists(SPLIT_CSV), f"Thiếu {SPLIT_CSV} — upload outputs/split_assignments.csv từ máy lên Drive."

[2026-10-28 12:51:41 UTC] TRAIN_CSV=/content/persuade_corpus_2.0_train.csv | sha256=f61319edd8bf16a982711ea0399fad59c05afaec05cdf0767f16a2c05c467e23


In [ ]:
# ===== BƯỚC 3: DỰNG ĐẦU VÀO L0 / L1 / PA =====
LABEL2ID = {"Ineffective": 0, "Adequate": 1, "Effective": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
CLASS_NAMES = ["Ineffective", "Adequate", "Effective"]
USECOLS = ["essay_id", "essay_id_comp", "discourse_id", "discourse_text", "discourse_type",
           "discourse_effectiveness", "discourse_start", "prompt_name"]


def excel_style_id(x):
    # Bản CSV gốc có essay_id toàn chữ số (vd 5408891152126); bản nhóm đã qua Excel thành 5.40889E+12.
    # Chỉ đổi ID toàn chữ số; ID dạng chữ/hex giữ nguyên (float("12E45") sẽ đọc nhầm).
    s = str(x).strip()
    return f"{float(s):.5E}" if s.isdigit() else s


def load_units(path):
    df = pd.read_csv(path, usecols=lambda c: c in USECOLS, dtype=str, low_memory=False)
    missing = set(USECOLS) - set(df.columns) - {"essay_id_comp"}
    assert not missing, f"{path} thiếu cột {missing}"
    if "essay_id_comp" not in df.columns:
        df["essay_id_comp"] = df["essay_id"]
    df = df.dropna(subset=["discourse_effectiveness", "discourse_text"]).copy()
    for c in ["discourse_text", "discourse_type", "essay_id", "essay_id_comp", "discourse_id"]:
        df[c] = df[c].astype(str).str.strip()
    df = df[df["discourse_text"].str.len() > 0].reset_index(drop=True)
    df["row_idx"] = np.arange(len(df))
    df["essay_key"] = df["essay_id"].map(excel_style_id)          # dự phòng khi map split
    df["prompt_name"] = df["prompt_name"].fillna("").astype(str).str.replace('"', "", regex=False).str.strip()
    df["label"] = df["discourse_effectiveness"].map(LABEL2ID)
    assert df["label"].notna().all(), "Có nhãn lạ ngoài 3 lớp"
    df["label"] = df["label"].astype(int)

    # Câu trước / sau trong cùng bài (theo essay_id_comp = ID thật, sắp theo discourse_start)
    df["_start"] = pd.to_numeric(df["discourse_start"], errors="coerce").fillna(0)
    s = df.sort_values(["essay_id_comp", "_start", "row_idx"], kind="mergesort")
    s["prev_text"] = s.groupby("essay_id_comp")["discourse_text"].shift(1).fillna("")
    s["next_text"] = s.groupby("essay_id_comp")["discourse_text"].shift(-1).fillna("")
    df = s.sort_values("row_idx").reset_index(drop=True)

    prev = df["prev_text"].where(df["prev_text"].str.len() > 0, "[NONE]")
    nxt = df["next_text"].where(df["next_text"].str.len() > 0, "[NONE]")
    # L0: y hệt script 02 | L1: y hệt script 05 | PA: L1 + tên đề ở đầu (thay đổi DUY NHẤT)
    df["text_L0"] = df["discourse_type"] + " [SEP] " + df["discourse_text"]
    df["text_L1"] = (df["discourse_type"] + " [SEP] Previous: " + prev + " [SEP] Target: " + df["discourse_text"]
                     + " [SEP] Next: " + nxt)
    df["text_PA"] = "Prompt: " + df["prompt_name"] + " [SEP] " + df["text_L1"]
    return df.drop(columns=["_start"])


full_train = load_units(TRAIN_CSV)
split_df = pd.read_csv(SPLIT_CSV, dtype=str)
split_map = dict(zip(split_df["essay_id"].str.strip(), split_df["split"].str.strip()))
split_map_norm = {excel_style_id(k): v for k, v in split_map.items()}
# Ưu tiên khớp ID nguyên văn (bản CSV của nhóm); chỉ dùng ID chuẩn hóa khi train CSV là bản gốc chưa qua Excel
full_train["split"] = full_train["essay_id"].map(split_map).fillna(full_train["essay_key"].map(split_map_norm))
n_unmapped = int(full_train["split"].isna().sum())
assert n_unmapped == 0, f"{n_unmapped} đơn vị không map được split"

train_df = full_train[full_train["split"] == "train"].reset_index(drop=True)
val_df = full_train[full_train["split"] == "val"].reset_index(drop=True)
log(f"Train: {len(train_df):,} đơn vị / {train_df.essay_id_comp.nunique():,} bài | "
    f"Val: {len(val_df):,} đơn vị / {val_df.essay_id_comp.nunique():,} bài")
if not QUICK:
    assert len(val_df) == 29102, f"Kỳ vọng 29.102 đơn vị val (như tuần 9–10), nhận {len(val_df):,}"
    assert len(train_df) == 115187, f"Kỳ vọng 115.187 đơn vị train, nhận {len(train_df):,}"
    log("Split khớp tuần 9–10: 115.187 train / 29.102 val.")

if QUICK:
    rng = np.random.RandomState(CFG["seed"])
    pick = lambda d, k: d[d.essay_id_comp.isin(rng.choice(d.essay_id_comp.unique(), k, replace=False))].reset_index(drop=True)
    train_df, val_df = pick(train_df, 60), pick(val_df, 20)
    log(f"QUICK: train {len(train_df)} / val {len(val_df)} đơn vị")

print("\nVí dụ đầu vào PA:\n", val_df["text_PA"].iloc[1][:400])
print("\nSố đề trong train:", train_df.prompt_name.nunique())

[2026-10-28 12:51:48 UTC] Train: 115,187 đơn vị / 12,475 bài | Val: 29,102 đơn vị / 3,119 bài
[2026-10-28 12:51:48 UTC] Split khớp tuần 9–10: 115.187 train / 29.102 val.

Ví dụ đầu vào PA:
 Prompt: Phones and driving [SEP] Position [SEP] Previous: People are debating whether if drivers should or should not use cell phones while operating a vehicle. Some people think that using a cell phone while operating a vehicle should be allowed because they try multi-tasking and have important calls while some are addicted and can't stay off their phone. Other people says that you shouldn't be a

Số đề trong train: 15


## Bước 4–5 — Train / nạp checkpoint

- **PA** luôn được train mới (đây là mô hình cải tiến của tuần 11).
- **L0, L1**: ưu tiên dùng checkpoint cũ (đúng model đã báo cáo trên validation tuần 9–10). Bước 6 sẽ kiểm tra checkpoint tái lập đúng Macro-F1 cũ (L0 = 0.5768, L1 = 0.5790). Nếu không còn checkpoint và `TRAIN_MISSING_BASELINES = True` thì train lại bằng đúng công thức — khi đó số validation có thể lệch nhẹ so với số cũ, phải ghi rõ trong báo cáo.
- Loss: CrossEntropy có trọng số lớp `balanced` (tính trên tập train), fp16, padding động, chọn epoch tốt nhất theo Macro-F1 validation — y hệt script 02/05.
- Checkpoint từng epoch được lưu lên Drive: nếu Colab ngắt, chạy lại từ đầu notebook là tự **resume**.

In [ ]:
# ===== HÀM DÙNG CHUNG: METRIC, TRAIN, SUY LUẬN =====
def metrics_dict(y, pred):
    f1c = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {
        "Macro-F1": f1_score(y, pred, labels=[0, 1, 2], average="macro", zero_division=0),
        "Accuracy": accuracy_score(y, pred),
        "Balanced Accuracy": balanced_accuracy_score(y, pred),
        "QWK": cohen_kappa_score(y, pred, weights="quadratic"),
        "Cohen Kappa": cohen_kappa_score(y, pred),
        "F1 Ineffective": f1c[0], "F1 Adequate": f1c[1], "F1 Effective": f1c[2],
        "Precision Ineffective": precision_score(y == 0, pred == 0, zero_division=0),
        "Recall Ineffective": recall_score(y == 0, pred == 0, zero_division=0),
    }


def hf_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    m = metrics_dict(labels, np.argmax(logits, -1))
    return {"macro_f1": m["Macro-F1"], "accuracy": m["Accuracy"], "f1_ineffective": m["F1 Ineffective"]}


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device, logits.dtype))(
            logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def is_model_dir(path):
    return os.path.isdir(path) and os.path.exists(os.path.join(path, "config.json")) and (
        glob.glob(os.path.join(path, "*.safetensors")) or glob.glob(os.path.join(path, "*.bin")))


def train_model(key, text_col, best_dir):
    # Train DeBERTa theo đúng công thức script 02/05, chỉ khác cột đầu vào. Resume được.
    enforce_reproducibility(CFG["seed"])
    ckpt_dir = best_dir + "_checkpoints"
    tok = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=True)

    def to_hf(d):
        ds = Dataset.from_pandas(d[[text_col, "label"]].rename(columns={text_col: "text", "label": "labels"}),
                                 preserve_index=False)
        return ds.map(lambda b: tok(b["text"], truncation=True, max_length=CFG["max_length"]),
                      batched=True, batch_size=2000, remove_columns=["text"])

    hf_train, hf_val = to_hf(train_df), to_hf(val_df)
    weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_df["label"].values)
    log(f"[{key}] class weights = {np.round(weights, 4).tolist()}")

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG["model_name"], num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"
    # transformers >= 5 bỏ warmup_ratio; warmup_steps dạng float < 1 được hiểu là tỷ lệ
    warmup = {"warmup_ratio": CFG["warmup_ratio"]} if "warmup_ratio" in ta_params else {"warmup_steps": CFG["warmup_ratio"]}
    args = TrainingArguments(**warmup, **{
        "output_dir": ckpt_dir, "learning_rate": CFG["lr"],
        "per_device_train_batch_size": CFG["batch_size"], "per_device_eval_batch_size": CFG["batch_size"] * 2,
        "gradient_accumulation_steps": CFG["grad_accum"], "num_train_epochs": CFG["epochs"],
        "fp16": USE_FP16, "bf16": USE_BF16, eval_key: "epoch", "save_strategy": "epoch",
        "load_best_model_at_end": True, "metric_for_best_model": "macro_f1", "greater_is_better": True,
        "save_total_limit": 1, "seed": CFG["seed"], "logging_steps": 100, "report_to": "none",
    })
    tr_params = inspect.signature(Trainer.__init__).parameters
    tok_kw = {"processing_class": tok} if "processing_class" in tr_params else {"tokenizer": tok}
    trainer = WeightedLossTrainer(model=model, args=args, train_dataset=hf_train, eval_dataset=hf_val,
                                  data_collator=DataCollatorWithPadding(tok), compute_metrics=hf_metrics,
                                  class_weights=torch.tensor(weights, dtype=torch.float), **tok_kw)

    last = get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None
    log(f"[{key}] Bắt đầu train ({'resume từ ' + last if last else 'từ đầu'}) | "
        f"{len(hf_train):,} train / {len(hf_val):,} val | {CFG['epochs']} epoch")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=last)
    minutes = (time.time() - t0) / 60
    trainer.save_model(best_dir)
    tok.save_pretrained(best_dir)
    epoch_logs = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
    with open(os.path.join(best_dir, "train_history.json"), "w", encoding="utf-8") as f:
        json.dump({"key": key, "text_col": text_col, "cfg": CFG, "train_minutes": round(minutes, 1),
                   "best_metric": trainer.state.best_metric, "epoch_evals": epoch_logs,
                   "class_weights": weights.tolist(), "gpu": GPU_NAME}, f, ensure_ascii=False, indent=2)
    for h in epoch_logs:
        log(f"[{key}] epoch {h.get('epoch')}: val Macro-F1={h['eval_macro_f1']:.4f} | "
            f"Ineff F1={h['eval_f1_ineffective']:.4f}")
    log(f"[{key}] Xong sau {minutes:.1f} phút. Best val Macro-F1={trainer.state.best_metric:.4f} -> {best_dir}")
    best = trainer.state.best_metric
    if not QUICK and (best is None or not np.isfinite(best) or best < 0.40):
        # Model sập (vd NaN -> đoán toàn 1 lớp, Macro-F1 ~0.03). Không dùng, xóa để lần sau train lại.
        shutil.rmtree(best_dir, ignore_errors=True)
        raise RuntimeError(f"[{key}] Train hỏng: best val Macro-F1={best}. Đã xóa {best_dir}; xem log loss/grad_norm.")
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    del trainer, model
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()


@torch.no_grad()
def predict_proba(model_dir, d, text_col):
    # Suy luận sắp theo độ dài để nhanh; trả về xác suất theo đúng thứ tự dòng của d.
    tok_src = model_dir if os.path.exists(os.path.join(model_dir, "tokenizer_config.json")) else CFG["model_name"]
    tok = AutoTokenizer.from_pretrained(tok_src, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE).eval()
    enc = tok(d[text_col].tolist(), truncation=True, max_length=CFG["max_length"])
    ids, att = enc["input_ids"], enc["attention_mask"]
    order = np.argsort([-len(x) for x in ids], kind="stable")
    collate = DataCollatorWithPadding(tok)
    probs = np.zeros((len(d), 3), dtype=np.float32)
    bs = CFG["eval_batch_size"]
    for i in tqdm(range(0, len(order), bs), desc=f"predict {os.path.basename(model_dir)}"):
        idx = order[i:i + bs]
        batch = collate([{"input_ids": ids[j], "attention_mask": att[j]} for j in idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_CUDA):
            logits = model(**batch).logits.float()
        probs[idx] = torch.softmax(logits, -1).cpu().numpy()
    del model
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()
    return probs


def essay_confusions(essays, y, pred):
    # Ma trận nhầm lẫn 3x3 cho từng bài -> mảng (n_bài, 9) để bootstrap nhanh
    codes, uniq = pd.factorize(essays)
    out = np.zeros((len(uniq), 9))
    np.add.at(out, (codes, y * 3 + pred), 1)
    return out


def macro_f1_from_cm(cm9):
    cm = cm9.reshape(-1, 3, 3)
    tp = np.diagonal(cm, axis1=1, axis2=2)
    denom = cm.sum(axis=2) + cm.sum(axis=1)
    f1 = np.divide(2 * tp, denom, out=np.zeros_like(tp, dtype=float), where=denom > 0)
    return f1.mean(axis=1)


def paired_bootstrap(essays, y, pred_a, pred_b, n=None, seed=42):
    # Bootstrap theo BÀI (không theo câu) cho chênh lệch Macro-F1 (b - a)
    n = n or CFG["n_bootstrap"]
    ca, cb = essay_confusions(essays, y, pred_a), essay_confusions(essays, y, pred_b)
    w = np.random.RandomState(seed).multinomial(len(ca), np.full(len(ca), 1 / len(ca)), size=n)
    diffs = macro_f1_from_cm(w @ cb) - macro_f1_from_cm(w @ ca)
    point = macro_f1_from_cm(cb.sum(0, keepdims=True))[0] - macro_f1_from_cm(ca.sum(0, keepdims=True))[0]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"delta_macro_f1": point, "ci95_low": lo, "ci95_high": hi, "p_delta_le_0": float((diffs <= 0).mean())}

In [ ]:
# ===== BƯỚC 4–5: TRAIN PA, TÌM / TRAIN L0 & L1 =====
SYSTEMS = {
    "L0": {"name": "3. DeBERTa-v3 Level 0", "text_col": "text_L0",
           "input": "Câu cô lập: [Type] [SEP] [Text]", "ref_val_macro_f1": 0.5768,
           "candidates": [os.path.join(OUT_DIR, "deberta_best_model")]},
    "L1": {"name": "4. DeBERTa-v3 Level 1 (Khóa)", "text_col": "text_L1",
           "input": "[Type] [SEP] Previous [SEP] Target [SEP] Next", "ref_val_macro_f1": 0.5790,
           "candidates": [os.path.join(OUT_DIR, "deberta_context_best_model")]},
    "PA": {"name": "5. DeBERTa-v3 Level 1 + Prompt-Aware", "text_col": "text_PA",
           "input": "Prompt: [Tên đề] [SEP] + đầu vào Level 1", "ref_val_macro_f1": None,
           "candidates": []},
}
for key, s in SYSTEMS.items():
    retrained_dir = os.path.join(P3_DIR, f"{key.lower()}_best_model")
    found = next((p for p in ([] if QUICK else s["candidates"]) if is_model_dir(p)), None)
    if found:
        s.update(model_dir=found, source="checkpoint cũ (tuần 9–10)")
    elif is_model_dir(retrained_dir):
        s.update(model_dir=retrained_dir, source="train ở notebook 07")
    elif key == "PA" or TRAIN_MISSING_BASELINES:
        train_model(key, s["text_col"], retrained_dir)
        s.update(model_dir=retrained_dir, source="train ở notebook 07")
    else:
        raise FileNotFoundError(f"Không thấy checkpoint {key} ở {s['candidates']} và TRAIN_MISSING_BASELINES=False")
    log(f"{key}: dùng {s['model_dir']} ({s['source']})")

Map:   0%|          | 0/115187 [00:00<?, ? examples/s]

Map:   0%|          | 0/29102 [00:00<?, ? examples/s]

[2026-10-28 12:52:10 UTC] [L0] class weights = [7.4787, 0.425, 1.9484]


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


[2026-10-28 15:00:30 UTC] [PA] Bắt đầu train (từ đầu) | 115,187 train / 29,102 val | 3 epoch


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,F1 Ineffective
1,0.700500,0.729417,0.547822,0.679472,0.351322
2,0.585000,0.737237,0.588227,0.744038,0.396969
3,0.513700,0.816065,0.581586,0.739262,0.383403


[2026-10-28 16:08:55 UTC] [PA] epoch 1.0: val Macro-F1=0.5478 | Ineff F1=0.3513
[2026-10-28 16:08:55 UTC] [PA] epoch 2.0: val Macro-F1=0.5882 | Ineff F1=0.3970
[2026-10-28 16:08:55 UTC] [PA] epoch 3.0: val Macro-F1=0.5816 | Ineff F1=0.3834
[2026-10-28 16:08:55 UTC] [PA] Xong sau 68.4 phút. Best val Macro-F1=0.5882 -> /content/outputs/week11_person3/pa_best_model


[2026-10-28 16:08:56 UTC] PA: dùng /content/outputs/week11_person3/pa_best_model (train ở notebook 07)


## Bước 6 — Validation (29.102 đơn vị): kiểm tra checkpoint & so PA với L1

Đây là bước **ra quyết định**. Xem kết quả ở đây rồi mới bật `RUN_OFFICIAL_TEST = True`.
- Nếu L0/L1 dùng checkpoint cũ mà Macro-F1 lệch số cũ > 0.002 → có thể nhầm checkpoint, dừng lại kiểm tra.
- Bootstrap theo bài cho PA − L1: nếu khoảng tin cậy 95% chứa 0 thì phải viết "chưa phân biệt được", giống cách viết L1 vs L0 ở tuần 10.

In [ ]:
# ===== BƯỚC 6: VALIDATION =====
val_rows, val_preds = [], {}
for key, s in SYSTEMS.items():
    vpath = os.path.join(P3_DIR, f"val_predictions_{key}.csv")
    if os.path.exists(vpath):
        probs = pd.read_csv(vpath)[["pred_proba_0", "pred_proba_1", "pred_proba_2"]].values
        assert len(probs) == len(val_df)
    else:
        probs = predict_proba(s["model_dir"], val_df, s["text_col"])
        out = val_df[["row_idx", "essay_id", "essay_id_comp", "discourse_id", "discourse_type", "prompt_name",
                      "discourse_start", "label"]].rename(columns={"label": "true_label"}).copy()
        out["pred_label"] = probs.argmax(1)
        for c in range(3):
            out[f"pred_proba_{c}"] = np.round(probs[:, c], 6)
        out.to_csv(vpath, index=False)
    pred = probs.argmax(1)
    val_preds[key] = pred
    m = metrics_dict(val_df["label"].values, pred)
    ref = s["ref_val_macro_f1"]
    note = ""
    if ref is not None and not QUICK:
        note = f"lệch {m['Macro-F1'] - ref:+.4f} so với số cũ {ref}"
        if s["source"].startswith("checkpoint") and abs(m["Macro-F1"] - ref) > 0.002:
            note += "  <-- CẢNH BÁO: checkpoint không tái lập số cũ!"
    val_rows.append({"Hệ thống": s["name"], "Nguồn model": s["source"],
                     **{k: round(v, 4) for k, v in m.items()}, "Ghi chú": note})

val_table = pd.DataFrame(val_rows)
val_table.to_csv(os.path.join(P3_DIR, "validation_L0_L1_PA.csv"), index=False)
display(val_table) if "display" in globals() else print(val_table.to_string(index=False))

essays_val = val_df["essay_id_comp"].values
y_val = val_df["label"].values
boot_val = pd.DataFrame([
    {"So sánh": "L1 − L0", **paired_bootstrap(essays_val, y_val, val_preds["L0"], val_preds["L1"])},
    {"So sánh": "PA − L1", **paired_bootstrap(essays_val, y_val, val_preds["L1"], val_preds["PA"])},
    {"So sánh": "PA − L0", **paired_bootstrap(essays_val, y_val, val_preds["L0"], val_preds["PA"])},
]).round(4)
boot_val.to_csv(os.path.join(P3_DIR, "validation_bootstrap_by_essay.csv"), index=False)
print("\nBootstrap theo bài trên validation (2.5%–97.5%):")
print(boot_val.to_string(index=False))
log("Validation xong. Xem bảng trên, rồi mới đặt RUN_OFFICIAL_TEST = True ở ô cấu hình.")

The tokenizer you are loading from '/content/outputs/week11_person3/l0_best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


predict l0_best_model:   0%|          | 0/455 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/outputs/week11_person3/l1_best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


predict l1_best_model:   0%|          | 0/455 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/outputs/week11_person3/pa_best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


predict pa_best_model:   0%|          | 0/455 [00:00<?, ?it/s]

                            Hệ thống         Nguồn model  Macro-F1  Accuracy  Balanced Accuracy    QWK  Cohen Kappa  F1 Ineffective  F1 Adequate  F1 Effective  Precision Ineffective  Recall Ineffective                          Ghi chú
               3. DeBERTa-v3 Level 0 train ở notebook 07    0.5718    0.7354             0.6107 0.4297       0.3612          0.3718       0.8218        0.5218                 0.3545              0.3908 lệch -0.0050 so với số cũ 0.5768
        4. DeBERTa-v3 Level 1 (Khóa) train ở notebook 07    0.5824    0.7429             0.6297 0.4463       0.3752          0.3870       0.8276        0.5325                 0.3380              0.4526  lệch +0.0034 so với số cũ 0.579
5. DeBERTa-v3 Level 1 + Prompt-Aware train ở notebook 07    0.5884    0.7442             0.6478 0.4578       0.3828          0.3970       0.8281        0.5400                 0.3241              0.5120                                 



Bootstrap theo bài trên validation (2.5%–97.5%):
So sánh  delta_macro_f1  ci95_low  ci95_high  p_delta_le_0
L1 − L0          0.0106    0.0022     0.0190         0.006
PA − L1          0.0060   -0.0004     0.0125         0.032
PA − L0          0.0166    0.0086     0.0245         0.000
[2026-10-28 16:10:17 UTC] Validation xong. Xem bảng trên, rồi mới đặt RUN_OFFICIAL_TEST = True ở ô cấu hình.


## Bước 7 — OFFICIAL TEST (chạy một lần duy nhất)

- Chấm **cùng lượt** L0, L1, PA trên toàn bộ tập test (93.677 đơn vị). RF và SVM đã được Người 1 và Người 2 chấm.
- Lưu dự đoán từng đơn vị lên Drive, sau đó ghi `OFFICIAL_TEST_LOCK.json`.
- Nếu đã có khóa, ô này **chỉ đọc lại** dự đoán đã lưu và tính lại số, **không suy luận lại**.
- Lưu ý khi viết báo cáo: DeBERTa train trên 80% train (115.187 đơn vị) và chọn epoch bằng 20% val. RF và SVM trong `run_*_official_test.py` train trên toàn bộ 144.289 đơn vị.

In [ ]:
# ===== BƯỚC 7: OFFICIAL TEST =====
LOCK_PATH = os.path.join(P3_DIR, "OFFICIAL_TEST_LOCK.json")
if not RUN_OFFICIAL_TEST and not os.path.exists(LOCK_PATH):
    raise SystemExit("RUN_OFFICIAL_TEST = False — dừng ở đây. Xem validation trước, rồi bật cờ và chạy lại ô này.")

TEST_CSV = find_test_csv()
test_df = load_units(TEST_CSV)
if QUICK:
    rng = np.random.RandomState(CFG["seed"])
    test_df = test_df[test_df.essay_id_comp.isin(rng.choice(test_df.essay_id_comp.unique(), 20, replace=False))]
    test_df = test_df.reset_index(drop=True)
log(f"TEST_CSV={TEST_CSV} | {len(test_df):,} đơn vị / {test_df.essay_id_comp.nunique():,} bài")
if not QUICK:
    assert len(test_df) == 93677, f"Kỳ vọng 93.677 đơn vị test (như RF/SVM), nhận {len(test_df):,}"

locked = os.path.exists(LOCK_PATH)
RUNTIME_PATH = os.path.join(P3_DIR, "test_runtimes.json")
runtimes = json.load(open(RUNTIME_PATH, encoding="utf-8")) if os.path.exists(RUNTIME_PATH) else {}
test_rows, test_preds = [], {}
for key, s in SYSTEMS.items():
    tpath = os.path.join(P3_DIR, f"test_predictions_{key}.csv")
    t0 = time.time()
    if os.path.exists(tpath):
        probs = pd.read_csv(tpath)[["pred_proba_0", "pred_proba_1", "pred_proba_2"]].values
        assert len(probs) == len(test_df)
        runtime = runtimes.get(key)
        log(f"[{key}] Đọc lại dự đoán test đã lưu: {tpath}")
    else:
        if locked:
            raise RuntimeError(f"Đã khóa test nhưng thiếu {tpath} — không suy luận lại để giữ kỷ luật test.")
        probs = predict_proba(s["model_dir"], test_df, s["text_col"])
        runtime = round(time.time() - t0, 1)
        out = test_df[["row_idx", "essay_id", "essay_id_comp", "discourse_id", "discourse_type", "prompt_name",
                       "discourse_start", "label"]].rename(columns={"label": "true_label"}).copy()
        out["pred_label"] = probs.argmax(1)
        for c in range(3):
            out[f"pred_proba_{c}"] = np.round(probs[:, c], 6)
        out.to_csv(tpath, index=False)
        runtimes[key] = runtime
        with open(RUNTIME_PATH, "w", encoding="utf-8") as f:
            json.dump(runtimes, f)
        log(f"[{key}] Suy luận test xong trong {runtime}s -> {tpath}")
    pred = probs.argmax(1)
    test_preds[key] = pred
    m = metrics_dict(test_df["label"].values, pred)
    test_rows.append({
        "Hệ thống": s["name"], "Kiểu mô hình & Đầu vào": s["input"],
        "Macro-F1": round(m["Macro-F1"], 4), "Accuracy (%)": round(m["Accuracy"] * 100, 2),
        "F1 Ineffective": round(m["F1 Ineffective"], 4), "F1 Adequate": round(m["F1 Adequate"], 4),
        "F1 Effective": round(m["F1 Effective"], 4), "Runtime (s)": runtime,
        "QWK": round(m["QWK"], 4), "Cohen Kappa": round(m["Cohen Kappa"], 4),
        "Recall Ineffective": round(m["Recall Ineffective"], 4), "N test": len(test_df),
        "Nguồn model": s["source"],
    })

if not locked:
    with open(LOCK_PATH, "w", encoding="utf-8") as f:
        json.dump({"locked_at": datetime.now(timezone.utc).isoformat(), "test_csv": TEST_CSV,
                   "test_sha256": sha256_of(TEST_CSV), "n_test": len(test_df),
                   "models": {k: s["model_dir"] for k, s in SYSTEMS.items()}, "cfg": CFG, "gpu": GPU_NAME,
                   "quick": QUICK}, f, ensure_ascii=False, indent=2)
    log(f"Đã ghi khóa test: {LOCK_PATH}")

test_table = pd.DataFrame(test_rows)
essays_test, y_test = test_df["essay_id_comp"].values, test_df["label"].values
boot_test = pd.DataFrame([
    {"So sánh": "L1 − L0", **paired_bootstrap(essays_test, y_test, test_preds["L0"], test_preds["L1"])},
    {"So sánh": "PA − L1", **paired_bootstrap(essays_test, y_test, test_preds["L1"], test_preds["PA"])},
    {"So sánh": "PA − L0", **paired_bootstrap(essays_test, y_test, test_preds["L0"], test_preds["PA"])},
]).round(4)

display(test_table) if "display" in globals() else print(test_table.to_string(index=False))
print("\nBootstrap theo bài trên OFFICIAL TEST:")
print(boot_test.to_string(index=False))

SystemExit: RUN_OFFICIAL_TEST = False — dừng ở đây. Xem validation trước, rồi bật cờ và chạy lại ô này.

In [ ]:
# ===== BƯỚC 8: XUẤT FILE CHO BẢNG MASTER =====
suffix = "_QUICK" if QUICK else ""
deberta_csv = os.path.join(OUT_DIR, f"official_test_deberta_results{suffix}.csv")
test_table.to_csv(deberta_csv, index=False)
boot_test.to_csv(os.path.join(OUT_DIR, f"official_test_deberta_bootstrap{suffix}.csv"), index=False)

# File theo đúng tên trong WEEK11 plan: kết quả mô hình cải tiến (val + test)
pa_val = val_table[val_table["Hệ thống"] == SYSTEMS["PA"]["name"]].assign(Tập="Validation (29.102)")
pa_test = test_table[test_table["Hệ thống"] == SYSTEMS["PA"]["name"]].assign(Tập=f"Official Test ({len(test_df):,})")
pd.concat([pa_val, pa_test], ignore_index=True).to_csv(
    os.path.join(OUT_DIR, f"improved_deberta_results{suffix}.csv"), index=False)

meta = {"finished_at": datetime.now(timezone.utc).isoformat(), "quick": QUICK, "gpu": GPU_NAME,
        "precision": "bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32",
        "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                     "transformers": transformers.__version__, "pandas": pd.__version__},
        "train_csv_sha256": train_sha, "cfg": CFG,
        "models": {k: {"dir": s["model_dir"], "source": s["source"]} for k, s in SYSTEMS.items()}}
with open(os.path.join(P3_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# Bảng 5 hệ thống ngay trong notebook nếu đã có file RF/SVM của Người 1 & 2
rows = []
for fname, label, kind in [("official_test_svm_actual_result.csv", "1. Linear SVM (TF-IDF Baseline)",
                            "Tuyến tính / Phân loại lề (TF-IDF 15k từ)"),
                           ("official_test_rf_actual_result.csv", "2. Random Forest (Feature Set B)",
                            "Rừng cây quyết định (Bag-of-words + Role + Len)")]:
    p = os.path.join(OUT_DIR, fname)
    if os.path.exists(p):
        r = pd.read_csv(p).iloc[0].to_dict()
        r.update({"Hệ thống": label, "Kiểu mô hình & Đầu vào": kind})
        rows.append(r)
    else:
        print(f"(Chưa có {fname} trên Drive — bảng dưới thiếu dòng này; chạy script_official_test_evaluation.py ở máy để ghép.)")
cols = ["Hệ thống", "Kiểu mô hình & Đầu vào", "Macro-F1", "Accuracy (%)", "F1 Ineffective", "F1 Adequate",
        "F1 Effective", "Runtime (s)"]
master = pd.concat([pd.DataFrame(rows), test_table], ignore_index=True)[cols]
master.to_csv(os.path.join(OUT_DIR, f"official_test_final_summary{suffix}.csv"), index=False)
print(master.to_string(index=False))
log(f"Đã ghi: {deberta_csv}, improved_deberta_results{suffix}.csv, official_test_final_summary{suffix}.csv")

if IN_COLAB:
    zip_base = os.path.join("/content", f"week11_person3_results{suffix}")
    shutil.make_archive(zip_base, "zip", P3_DIR)
    print(f"Đã nén {P3_DIR} -> {zip_base}.zip (không gồm checkpoint ở thư mục khác). "
          "Tải về rồi chép các CSV trong outputs/ vào thư mục outputs/ trên máy.")

NameError: name 'test_table' is not defined

## Sau khi chạy xong

1. **File → Save a copy in Drive** (hoặc Download .ipynb) để giữ output các cell. Chép notebook đã chạy vào repo thành `week11_person3_full/07_person3_prompt_aware_official_test_output.ipynb`.
2. Tải về máy các file trong `MyDrive/MLA_Dataset/outputs/`: `official_test_deberta_results.csv`, `official_test_deberta_bootstrap.csv`, `improved_deberta_results.csv`, `official_test_final_summary.csv`, và thư mục `week11_person3/` (dự đoán từng đơn vị + log + khóa test).
3. Trên máy: `python script_official_test_evaluation.py`. Script giờ đọc số DeBERTa thật từ `outputs/official_test_deberta_results.csv`, không còn dùng số gõ cứng.
4. Báo cáo Mục 1.3: thay mọi số "kỳ vọng/tiềm năng" bằng số thật, kèm bootstrap PA − L1 và L1 − L0 trên test.